# Variational Autoencoders: latent capacity, architecture, and sample quality

**Assignment notebook — synthetic Two Moons → conditional MNIST.**

This notebook implements a **VAE from scratch in PyTorch**. It compares latent dimensions, model capacity and optionally KL weighting, and analyzes **prior-generated samples**, not just reconstruction/test loss.

### Research questions
1. How does latent dimension (1/2/4/8 for synthetic; 2/8/16/32 for MNIST) affect reconstruction and generation?
2. How does architecture capacity change the trade-off?
3. Are reconstructions sharp but independent prior samples poor? How does KL regularization relate?

**Design:** fixed-seed train/validation/test partitions; validation for model comparisons; a separate final test evaluation. Figures are produced by training the models here (no precomputed results). All metric values shown in outputs depend on your run.

**Recommended workflow:** run all cells with `QUICK_RUN=True` first, then set `QUICK_RUN=False` for final results. Change only one factor at a time. Save important images/tables under `vae_outputs/`.

## 0. Environment and reproducibility

Install dependencies if needed with: `%pip install torch torchvision scikit-learn matplotlib pandas tqdm`.

**Data download:** The MNIST section requires internet access on its first execution to download the dataset. `torchvision` must be compatible with your installed PyTorch version.

**Compute note:** Even the final experiments train multiple models; CUDA makes them much faster. The default MNIST subset keeps CPU execution practical. For a full MNIST experiment, set `MNIST_TRAIN_LIMIT=None` etc.

In [ ]:
from pathlib import Path
import random, copy, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Subset

SEED = 42
QUICK_RUN = True  # Change to False for full report experiments
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('vae_outputs'); OUT.mkdir(exist_ok=True)

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all()
print('Device:', DEVICE, '| Quick run:', QUICK_RUN)

## 1. Shared implementation

The encoder predicts `mu` and `logvar`; the **reparameterization trick** computes `z = mu + eps * exp(0.5 * logvar)` so gradients can flow. A standard normal prior `N(0,I)` supports generating samples independently of the training set.

- Synthetic 2D outputs: Gaussian decoder with **fixed unit variance**, giving reconstruction term `0.5 * Σ(x−x̂)²` per observation (constants omitted).
- MNIST: Bernoulli pixel likelihood implemented as **binary cross-entropy with logits**. MNIST pixels are scaled to [0,1]; BCE is a common baseline, though pixels are grayscale rather than truly binary.
- Loss is **reconstruction + β × KL** summed across features, averaged over batch, allowing apples-to-apples comparison *within* each dataset. Synthetic and MNIST losses should **not** be compared directly.
- `condition_dim=10` produces a conditional VAE for MNIST, with one-hot label attached to **both** encoder and decoder.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=2, hidden=(128,64), condition_dim=0):
        super().__init__()
        self.input_dim, self.latent_dim, self.condition_dim = input_dim, latent_dim, condition_dim
        self.encoder = self._mlp(input_dim + condition_dim, hidden)
        h = hidden[-1] if hidden else input_dim + condition_dim
        self.to_mu = nn.Linear(h, latent_dim)
        self.to_logvar = nn.Linear(h, latent_dim)
        self.decoder = self._mlp(latent_dim + condition_dim, tuple(reversed(hidden)), input_dim)

    @staticmethod
    def _mlp(in_dim, widths, out_dim=None):
        layers=[]; last=in_dim
        for width in widths:
            layers.extend([nn.Linear(last,width), nn.ReLU()]); last=width
        if out_dim is not None: layers.append(nn.Linear(last,out_dim))
        return nn.Sequential(*layers) if layers else nn.Identity()

    def _cond(self, y, batch, device):
        if not self.condition_dim: return None
        if y is None: raise ValueError('Class labels required for conditional model')
        return torch.nn.functional.one_hot(y.long(), self.condition_dim).to(device=device, dtype=torch.float32)

    def encode(self, x, y=None):
        x = x.reshape(x.shape[0],-1)
        cond = self._cond(y, len(x), x.device)
        h = self.encoder(torch.cat([x,cond],dim=1) if cond is not None else x)
        return self.to_mu(h), self.to_logvar(h)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5*logvar)

    def decode(self, z, y=None):
        cond = self._cond(y, len(z), z.device)
        return self.decoder(torch.cat([z,cond],dim=1) if cond is not None else z)

    def forward(self, x, y=None):
        mu, logvar = self.encode(x,y)
        return self.decode(self.reparameterize(mu,logvar),y), mu, logvar

    @torch.no_grad()
    def sample(self, n, y=None, seed=123):
        # Independent fixed-seed sampling without perturbing global training RNG
        gen = torch.Generator(device=next(self.parameters()).device).manual_seed(seed)
        z = torch.randn(n, self.latent_dim, generator=gen, device=next(self.parameters()).device)
        if y is not None: y = y.to(z.device)
        return self.decode(z,y)


def vae_loss(out, x, mu, logvar, kind='gaussian', beta=1.):
    target=x.reshape(x.shape[0],-1)
    if kind=='gaussian':
        recon = 0.5 * ((out-target)**2).sum(dim=1).mean()
    elif kind=='bernoulli':
        recon = nn.functional.binary_cross_entropy_with_logits(out,target,reduction='none').sum(dim=1).mean()
    else: raise ValueError(kind)
    kl = (-0.5*(1+logvar-mu.square()-logvar.exp()).sum(dim=1)).mean()
    return recon+beta*kl, recon, kl


def evaluate(model, loader, kind, beta=1):
    model.eval(); totals=np.zeros(3); count=0
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            out,mu,lv=model(x,y if model.condition_dim else None)
            loss,recon,kl=vae_loss(out,x,mu,lv,kind,beta)
            totals += np.array([loss.item(),recon.item(),kl.item()])*len(x)
            count+=len(x)
    return dict(zip(['loss','recon','kl'],totals/count))


def fit(model, train_loader, val_loader, kind, epochs=20, lr=1e-3, beta=1, name='model'):
    model=model.to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    records=[]; best=float('inf'); best_state=None; best_epoch=0
    for ep in range(1,epochs+1):
        model.train(); sums=np.zeros(3); n=0
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            out,mu,lv=model(x,y if model.condition_dim else None)
            loss,recon,kl=vae_loss(out,x,mu,lv,kind,beta)
            loss.backward(); opt.step()
            sums+=np.array([loss.item(),recon.item(),kl.item()])*len(x); n+=len(x)
        val=evaluate(model,val_loader,kind,beta)
        records.append(dict(epoch=ep,train_loss=sums[0]/n,train_recon=sums[1]/n,train_kl=sums[2]/n,
                            val_loss=val['loss'],val_recon=val['recon'],val_kl=val['kl']))
        if val['loss'] < best:
            best=val['loss']; best_epoch=ep; best_state=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)  # restore best validation checkpoint; no test-set tuning
    return model, pd.DataFrame(records), best_epoch


def plot_history(history,title,filename=None):
    fig,axes=plt.subplots(1,3,figsize=(13,3.2))
    for ax,metric in zip(axes,['loss','recon','kl']):
        ax.plot(history.epoch,history['train_'+metric],label='train')
        ax.plot(history.epoch,history['val_'+metric],label='validation')
        ax.set(title=metric.upper(),xlabel='Epoch'); ax.grid(alpha=.2)
    axes[0].legend(); fig.suptitle(title); fig.tight_layout()
    if filename: fig.savefig(OUT/filename,dpi=150)
    plt.show()

## 2. Synthetic task: noisy Two Moons

Split data into 70% training, 15% validation and 15% testing. **Standardize using training data only**, to avoid leakage. Two Moons is not distributed like a Gaussian; the key question is whether prior-generated points recover both curved clusters, without filling the empty central gap.

Because an unconditional continuous decoder maps a Gaussian latent prior to a 2D distribution, it can struggle to reproduce disconnected-looking modes. Discuss this limitation if you see bridges or blurred clusters.

In [ ]:
X_raw, cluster = make_moons(n_samples=6000,noise=0.07,random_state=SEED)
ids=np.arange(len(X_raw))
tr_ids, hold_ids=train_test_split(ids,test_size=.30,random_state=SEED,stratify=cluster)
va_ids, te_ids=train_test_split(hold_ids,test_size=.50,random_state=SEED,stratify=cluster[hold_ids])
mean,std=X_raw[tr_ids].mean(0), X_raw[tr_ids].std(0)
X=((X_raw-mean)/std).astype('float32')

def synthetic_loader(indices,shuffle=False):
    # Dummy targets, ignored for unconditional VAE
    ds=TensorDataset(torch.from_numpy(X[indices]),torch.zeros(len(indices),dtype=torch.long))
    return DataLoader(ds,batch_size=256,shuffle=shuffle)
moons_train=synthetic_loader(tr_ids,True)
moons_val=synthetic_loader(va_ids)
moons_test=synthetic_loader(te_ids)
fig,ax=plt.subplots(figsize=(5.5,4))
ax.scatter(X_raw[tr_ids,0],X_raw[tr_ids,1],c=cluster[tr_ids],s=3,cmap='coolwarm',alpha=.7)
ax.set(title='Two Moons — training data (visualization colors only)',xlabel='x1',ylabel='x2')
plt.show()

### 2A. Compare latent dimensions (primary experiment)

Hold hidden layers `(64, 64)`, learning rate and KL weight fixed; vary only latent size. Report both numeric metrics and generated samples. Low-dimensional synthetic data does **not** imply that larger latent dimensions must be superior.

In [ ]:
SYN_DIMS=[1,2] if QUICK_RUN else [1,2,4,8]
SYN_EPOCHS=8 if QUICK_RUN else 45
syn_models={}; syn_hist={}; syn_rows=[]
for dim in SYN_DIMS:
    seed_all(SEED)
    m=VAE(input_dim=2,latent_dim=dim,hidden=(64,64))
    m,h,best_ep=fit(m,moons_train,moons_val,'gaussian',epochs=SYN_EPOCHS,name=f'syn-z{dim}')
    syn_models[dim]=m; syn_hist[dim]=h
    metrics=evaluate(m,moons_val,'gaussian')
    syn_rows.append({'latent_dim':dim,'best_epoch':best_ep,**{'val_'+k:v for k,v in metrics.items()}})
syn_result=pd.DataFrame(syn_rows)
display(syn_result.round(4))
syn_result.to_csv(OUT/'synthetic_latent_validation.csv',index=False)
plot_history(syn_hist[SYN_DIMS[-1]],f'Two Moons: latent dimension {SYN_DIMS[-1]}','synthetic_training.png')

In [ ]:
fig,axes=plt.subplots(1,len(SYN_DIMS)+1,figsize=(4*(len(SYN_DIMS)+1),3.7),sharex=True,sharey=True)
axes[0].scatter(X[:,0],X[:,1],s=2,alpha=.25)
axes[0].set_title('Real standardized data')
for ax,dim in zip(axes[1:],SYN_DIMS):
    pts=syn_models[dim].sample(2500,seed=999).cpu().numpy()
    ax.scatter(pts[:,0],pts[:,1],s=2,alpha=.25)
    ax.set_title(f'Generated from prior: z={dim}')
for ax in axes: ax.set(xlabel='x1',ylabel='x2'); ax.set_xlim(-2.8,2.8); ax.set_ylim(-2.8,2.8)
fig.tight_layout(); fig.savefig(OUT/'synthetic_generated_by_latent.png',dpi=160); plt.show()

**Questions to answer after training:** Which latent size best preserves the separated arcs? Does a larger latent improve loss but place points in the gap? Are the generated distributions too compact, broad or connected? Compare the figures with the validation metrics.

### 2B. Architecture comparison (keep latent dimension fixed)

Compare both depth and width at `z=2`. These runs are deliberately separate from the latent sweep; isolate one experimental variable.

In [ ]:
ARCHS={'small_1x32':(32,), 'medium_2x64':(64,64), 'large_3x128':(128,128,128)}
syn_arch_rows=[]; syn_arch_models={}
for name,hidden in (list(ARCHS.items())[:2] if QUICK_RUN else ARCHS.items()):
    seed_all(SEED)
    m=VAE(2,2,hidden=hidden)
    m,h,ep=fit(m,moons_train,moons_val,'gaussian',epochs=SYN_EPOCHS)
    syn_arch_models[name]=m
    syn_arch_rows.append({'architecture':name,'params':sum(p.numel() for p in m.parameters()),'best_epoch':ep,
                          **{'val_'+k:v for k,v in evaluate(m,moons_val,'gaussian').items()}})
syn_arch_df=pd.DataFrame(syn_arch_rows)
display(syn_arch_df.round(4)); syn_arch_df.to_csv(OUT/'synthetic_architecture_validation.csv',index=False)
fig,axes=plt.subplots(1,len(syn_arch_models),figsize=(4*len(syn_arch_models),3.6),squeeze=False)
for ax,(name,m) in zip(axes.ravel(),syn_arch_models.items()):
    pts=m.sample(2500,seed=999).cpu().numpy(); ax.scatter(pts[:,0],pts[:,1],s=2,alpha=.35)
    ax.set(title=name,xlim=(-2.8,2.8),ylim=(-2.8,2.8))
fig.tight_layout(); fig.savefig(OUT/'synthetic_arch_generated.png',dpi=150); plt.show()

## 3. Real dataset: Conditional MNIST

The CVAE sees the digit label `y` in **both** encoder and decoder; after training we can specify an intended class when sampling. The model is fully-connected (MLP), no CNN required. For a fair comparison, we compare *conditional VAEs* at different latent dimensions rather than comparing a conditional model against an unconditional model and attributing all changes to `z`.

**MNIST split:** official 60,000 train / 10,000 test from torchvision. From the official training partition, reserve 10% for validation using a fixed stratified split. Optionally restrict the resulting partitions to seeded, stratified subsets. Do **not** use test results for choosing the latent dimension or architecture.

In [ ]:
# torchvision imported here so the synthetic experiments can run even if it is unavailable.
try:
    from torchvision.datasets import MNIST
    from torchvision import transforms
except Exception as exc:
    raise RuntimeError('Install matching torch + torchvision versions: %pip install torch torchvision') from exc

mnist_transform=transforms.ToTensor()
mnist_train_base=MNIST(root='data',train=True,download=True,transform=mnist_transform)
mnist_test_base=MNIST(root='data',train=False,download=True,transform=mnist_transform)
labels=np.asarray(mnist_train_base.targets)
train_ids,val_ids=train_test_split(np.arange(len(labels)),test_size=.10,random_state=SEED,stratify=labels)
test_ids=np.arange(len(mnist_test_base))

def stratified_limit(indices,all_labels,limit,seed=SEED):
    if limit is None or len(indices)<=limit: return indices
    idx,_=train_test_split(indices,train_size=limit,random_state=seed,stratify=all_labels[indices])
    return idx

# Modify these limits to use more MNIST data. QUICK_RUN takes tiny slices for smoke testing.
MNIST_TRAIN_LIMIT=2000 if QUICK_RUN else 12000
MNIST_VAL_LIMIT=500 if QUICK_RUN else 2000
MNIST_TEST_LIMIT=500 if QUICK_RUN else 2000
train_ids=stratified_limit(train_ids,labels,MNIST_TRAIN_LIMIT)
val_ids=stratified_limit(val_ids,labels,MNIST_VAL_LIMIT)
test_ids=stratified_limit(test_ids,np.asarray(mnist_test_base.targets),MNIST_TEST_LIMIT)

mn_train=DataLoader(Subset(mnist_train_base,train_ids.tolist()),batch_size=128,shuffle=True)
mn_val=DataLoader(Subset(mnist_train_base,val_ids.tolist()),batch_size=256,shuffle=False)
mn_test=DataLoader(Subset(mnist_test_base,test_ids.tolist()),batch_size=256,shuffle=False)
print('MNIST:',len(train_ids),'train /',len(val_ids),'validation /',len(test_ids),'test')

### 3A. Main experiment: conditional VAE latent dimension sweep

Keep model architecture `(128,64)` and β=1 constant; sweep `z=2,8,16,32`. MNIST uses the **sum of pixel BCEs**, so reported reconstruction losses are per-image totals (not mean per pixel). Interpret magnitudes only against matching MNIST runs.

In [ ]:
MN_DIMS=[2,8] if QUICK_RUN else [2,8,16,32]
MN_EPOCHS=3 if QUICK_RUN else 25
mn_models={}; mn_hist={}; mn_rows=[]
for dim in MN_DIMS:
    seed_all(SEED)
    model=VAE(input_dim=784,latent_dim=dim,hidden=(128,64),condition_dim=10)
    model,h,ep=fit(model,mn_train,mn_val,'bernoulli',epochs=MN_EPOCHS)
    mn_models[dim]=model; mn_hist[dim]=h
    mn_rows.append({'latent_dim':dim,'best_epoch':ep,'params':sum(p.numel() for p in model.parameters()),
                    **{'val_'+k:v for k,v in evaluate(model,mn_val,'bernoulli').items()}})
mn_df=pd.DataFrame(mn_rows)
display(mn_df.round(4)); mn_df.to_csv(OUT/'mnist_latent_validation.csv',index=False)
plot_history(mn_hist[MN_DIMS[-1]],f'CVAE MNIST z={MN_DIMS[-1]}','mnist_training.png')

### 3B. Generated images — the key assignment figure

Every row is a requested digit (0–9); each column is an **independent latent prior draw**. For each model, use the same random seed so comparisons are easier. A conditional model may still produce illegible digits, blurred outputs, or repeated handwriting styles. Notice whether requested labels are recognizable and samples within each row vary.

In [ ]:
@torch.no_grad()
def plot_generated_digits(model, title, fname=None, n_per_class=7, seed=345):
    model.eval()
    y=torch.arange(10,device=DEVICE).repeat_interleave(n_per_class)
    out=model.sample(len(y),y,seed=seed).sigmoid().reshape(-1,28,28).cpu().numpy()
    fig,axes=plt.subplots(10,n_per_class,figsize=(n_per_class*1.05,10*1.0))
    for idx,ax in enumerate(axes.ravel()):
        ax.imshow(out[idx],cmap='gray',vmin=0,vmax=1); ax.axis('off')
        if idx % n_per_class==0: ax.set_ylabel(str(idx//n_per_class),rotation=0,labelpad=12)
    fig.suptitle(title); fig.tight_layout()
    if fname: fig.savefig(OUT/fname,dpi=160,bbox_inches='tight')
    plt.show()

for dim in MN_DIMS:
    plot_generated_digits(mn_models[dim],f'CVAE generated digits: latent z={dim}',f'mnist_generated_z{dim}.png')

### 3C. Reconstructions versus independent generations

**Reconstructions** start from held-out real MNIST images, encode them and use the posterior mean (`mu`) for a stable reconstruction. **Generations** above start from randomly sampled `z ~ N(0,I)`. A VAE can look better at reconstruction than prior sampling; don't conflate them.

In [ ]:
@torch.no_grad()
def plot_reconstruction(model,loader,title,fname=None,n=10):
    model.eval(); x,y=next(iter(loader)); x,y=x[:n].to(DEVICE),y[:n].to(DEVICE)
    mu,_=model.encode(x,y if model.condition_dim else None)
    out=model.decode(mu,y if model.condition_dim else None).sigmoid().reshape(-1,28,28).cpu().numpy()
    fig,axes=plt.subplots(2,n,figsize=(n*1.12,2.7))
    for j in range(n):
        axes[0,j].imshow(x[j,0].cpu(),cmap='gray',vmin=0,vmax=1)
        axes[1,j].imshow(out[j],cmap='gray',vmin=0,vmax=1)
        for i in range(2): axes[i,j].axis('off')
    axes[0,0].set_ylabel('Original',rotation=90); axes[1,0].set_ylabel('Recon',rotation=90)
    fig.suptitle(title); fig.tight_layout()
    if fname: fig.savefig(OUT/fname,dpi=150,bbox_inches='tight')
    plt.show()
for dim in MN_DIMS:
    plot_reconstruction(mn_models[dim],mn_val,f'MNIST reconstructions z={dim}',f'mnist_reconstruction_z{dim}.png')

### 3D. Latent-space visualization for `z=2`

Plot posterior means for held-out validation images, colored by digit. Because this is a **conditional** VAE, digit identity can be conveyed by the conditioning label; separation into 10 distinct latent clusters is **not necessary or expected**. Overlap is not necessarily a failure.

In [ ]:
@torch.no_grad()
def plot_latent_2d(model,loader):
    assert model.latent_dim==2
    model.eval(); pts=[]; labs=[]
    for x,y in loader:
        mu,_=model.encode(x.to(DEVICE),y.to(DEVICE))
        pts.append(mu.cpu().numpy()); labs.append(y.numpy())
    p=np.concatenate(pts); y=np.concatenate(labs)
    fig,ax=plt.subplots(figsize=(6,5))
    sc=ax.scatter(p[:,0],p[:,1],c=y,cmap='tab10',s=7,alpha=.7,vmin=-.5,vmax=9.5)
    fig.colorbar(sc,ax=ax,label='Label'); ax.set(xlabel='z1 mean',ylabel='z2 mean',title='CVAE posterior means (validation set)')
    fig.tight_layout(); fig.savefig(OUT/'mnist_latent_z2.png',dpi=160); plt.show()
plot_latent_2d(mn_models[2],mn_val)

## 4. Architecture ablation on MNIST (optional but recommended)

Hold `z=8` fixed. Compare width/depth with matched training protocol. Differences in parameter count are expected; report them. Each figure shows genuine new samples.

In [ ]:
MN_ARCHS={'small_1x64':(64,), 'medium_2layers':(128,64), 'large_3layers':(256,128,64)}
mn_arch_models={}; mn_arch_rows=[]
for name,hidden in (list(MN_ARCHS.items())[:2] if QUICK_RUN else MN_ARCHS.items()):
    seed_all(SEED)
    m=VAE(784,8,hidden=hidden,condition_dim=10)
    m,h,ep=fit(m,mn_train,mn_val,'bernoulli',epochs=MN_EPOCHS)
    mn_arch_models[name]=m
    mn_arch_rows.append({'architecture':name,'params':sum(p.numel() for p in m.parameters()),'best_epoch':ep,
                         **{'val_'+k:v for k,v in evaluate(m,mn_val,'bernoulli').items()}})
mn_arch_df=pd.DataFrame(mn_arch_rows)
display(mn_arch_df.round(4)); mn_arch_df.to_csv(OUT/'mnist_architecture_validation.csv',index=False)
for name,model in mn_arch_models.items():
    plot_generated_digits(model,f'Architecture {name}, z=8',f'mnist_arch_{name}.png',n_per_class=5)

## 5. β-VAE experiment (optional extension)

Hold `z=8` and architecture fixed. Compare KL weighting of `0.1`, `1.0`, and `4.0`. **Important:** because the objective changes with beta, avoid comparing the *combined* total loss across beta values as though it were the same metric; instead separately compare unweighted reconstruction term, unweighted KL term, and generated sample quality. This section is optional but particularly useful if generated images are blurry or prior samples are weak.

In [ ]:
RUN_BETA_EXPERIMENT = not QUICK_RUN  # May switch off if compute is limited
beta_models={}; beta_rows=[]
if RUN_BETA_EXPERIMENT:
    for beta in [0.1,1.0,4.0]:
        seed_all(SEED)
        m=VAE(784,8,hidden=(128,64),condition_dim=10)
        m,h,ep=fit(m,mn_train,mn_val,'bernoulli',epochs=MN_EPOCHS,beta=beta)
        beta_models[beta]=m
        metrics=evaluate(m,mn_val,'bernoulli',beta=1.0) # same unweighted recon + KL for diagnostic comparison
        beta_rows.append({'beta':beta,'best_epoch':ep,'val_recon':metrics['recon'],'val_kl':metrics['kl']})
        plot_generated_digits(m,f'beta={beta}, z=8',f'mnist_beta_{beta}.png',n_per_class=5)
    beta_df=pd.DataFrame(beta_rows)
    display(beta_df.round(4)); beta_df.to_csv(OUT/'mnist_beta_validation.csv',index=False)
else:
    print('Skipped optional beta experiment. Set RUN_BETA_EXPERIMENT=True to run it.')

## 6. Final held-out test evaluation

**Do this after selecting configurations based on validation and sample inspection.** For this assignment, evaluate each configuration on the same held-out test partition once you finalize experiments. Do not use test values to choose hyperparameters. Generated sample quality is assessed visually here; numerical VAE loss alone is not a full generative-quality measure.

In [ ]:
syn_test_df=pd.DataFrame([{'latent_dim':dim,**evaluate(m,moons_test,'gaussian')} for dim,m in syn_models.items()])
mn_test_df=pd.DataFrame([{'latent_dim':dim,**evaluate(m,mn_test,'bernoulli')} for dim,m in mn_models.items()])
print('Two Moons held-out test (per-example):'); display(syn_test_df.round(4))
print('Conditional MNIST held-out test (per-image):'); display(mn_test_df.round(4))
syn_test_df.to_csv(OUT/'synthetic_latent_test.csv',index=False)
mn_test_df.to_csv(OUT/'mnist_latent_test.csv',index=False)

## 7. Writing the analysis

Use the generated tables and images to answer these prompts **with concrete observed values and references to your plots**. Don't assert these patterns happened until your run supports them.

### Suggested results outline

**Synthetic (Two Moons)**
- Compare validation/test reconstruction and KL across `z=1,2,4,8`.
- Identify which models cover both moons. Do generations spill into the gap or collapse to a narrow region?
- At `z=2`, do wider/deeper models meaningfully improve generated geometry?

**MNIST (CVAE)**
- Compare validation/test reconstruction and KL across `z=2,8,16,32`.
- Contrast prior samples *and* held-out reconstructions across latent sizes.
- In label-conditioned generations, are all 10 requested digits recognizable? Are there multiple writing styles per label?
- Are improvements in reconstruction accompanied by improvements in independent generation? Where do they diverge?
- Discuss model size/parameter counts and compute cost; if β experiment is enabled, discuss the tradeoff of sharper reconstructions vs latent regularization.

**Limitations / methodology**
- Two Moons generated samples are judged visually; avoid claiming rigorous distribution match from a scatter plot alone.
- MNIST samples are judged visually and subjectively unless additional quantitative sample-quality evaluation is added.
- The default notebook uses stratified MNIST subsets, not the entire dataset; clearly state the selected sample counts.
- Conditioning makes class identity available to the decoder; thus CVAE latent-space clustering should not be interpreted as unconditional class discovery.
- Repeating key runs with several random seeds would strengthen claims; for a shorter assignment, acknowledge single-seed uncertainty.

### Example discussion template (replace brackets with actual results)
> On Two Moons, increasing the latent dimension from [A] to [B] changed validation reconstruction loss from [X] to [Y]. However, the generated scatter plots showed [specific geometric effect], indicating that lower reconstruction error did not necessarily mean a closer match to the input distribution. On conditional MNIST, the generated images for z=[...] appeared [description], while the reconstructions [description]. Taken together, these results suggest [measured trade-off], subject to the limitation of [single-seed/subset/visual scoring].

### Submission checklist
- [ ] `QUICK_RUN=False`; rerun all cells from a fresh kernel.
- [ ] Review real generated sample plots and label them clearly.
- [ ] State split sizes, epochs, architecture, beta and seed.
- [ ] Include comparison of at least 3–4 latent sizes per dataset.
- [ ] Discuss sample quality and diversity, not just numerical errors.
- [ ] Cite the use of any AI assistance as required by your course policy.